In [1]:
import pandas as pd

netid = "lsantucc"
df = pd.read_csv(f"{netid}_project_summary.csv", sep=";")
df.columns = ['project', 'commit', 'author', 'time', 'message']
df['Month'] = pd.to_datetime(df['time'], unit='s').dt.strftime('%Y-%m')

stats = pd.read_csv(f"{netid}_project_stats.csv", sep=";")

parts = []
for _, row in stats.iterrows():
    prj, gs, ge = row['Project'], row['LongestGapStart'], row['LongestGapEnd']
    if pd.isna(gs) or gs == 'None':
        continue   # aol_moloch has no gap
    d = df[df['project'] == prj].sort_values('time')
    pre = d[d['Month'] < gs].tail(10).copy()    # last 10 before gap
    post = d[d['Month'] > ge].head(10).copy()   # first 10 after gap
    pre.insert(0, 'pre/post', 'pre')
    post.insert(0, 'pre/post', 'post')
    parts += [pre, post]

gap = pd.concat(parts).drop(columns='Month')
gap.to_csv(f"{netid}_project_gap_commits.csv", sep=";", index=False)
gap.groupby(['project', 'pre/post']).size()   # sanity check: should be ≤10 each

project                       pre/post
acclab_dabestr                post        10
                              pre         10
covid19datahub_covid19dev     post        10
                              pre         10
echonet_dynamic               post        10
                              pre         10
eendebakpt_oapackage          post        10
                              pre         10
egri-nagy_sgpdec              post        10
                              pre         10
ncoudray_deeppath             post        10
                              pre         10
opengeospatial_ogc-geosparql  post        10
                              pre         10
smearle_gym-city              post        10
                              pre         10
wojdyr_fityk                  post        10
                              pre         10
dtype: int64

In [ ]:
for prj in gap['project'].unique():
    for side in ['pre', 'post']:
        print(f"\n===== {prj} | {side} =====")
        for m in gap[(gap['project'] == prj) & (gap['pre/post'] == side)]['message']:
            print(" -", str(m).strip()[:120])

In [ ]:
stats['Currentstatus'] = stats['lastGHCommitDate'].apply(
    lambda d: 'Inactive' if pd.to_datetime(d) < pd.Timestamp('2025-04-01') else 'Active')

for prj in stats['Project']:
    recent = df[df['project'] == prj].sort_values('time').tail(10)
    print(f"\n===== {prj} | recent =====")
    for m in recent['message']:
        print(" -", str(m).strip()[:120])

In [4]:
for prj in gap['project'].unique():
    g = gap[gap['project'] == prj]
    pre_a = set(g[g['pre/post'] == 'pre']['author'])
    post_a = set(g[g['pre/post'] == 'post']['author'])
    print(f"\n{prj}: returning={len(pre_a & post_a)}, new={len(post_a - pre_a)}")
    print("   new:", post_a - pre_a)


covid19datahub_covid19dev: returning=1, new=0
   new: set()

echonet_dynamic: returning=1, new=2
   new: {'Ashwin Nayak <anayak247@gmail.com>', 'David Ouyang <david.ouyang@gmail.com>'}

opengeospatial_ogc-geosparql: returning=2, new=0
   new: set()

egri-nagy_sgpdec: returning=2, new=2
   new: {'Max Horn <max@quendi.de>', 'Chrystopher L. Nehaniv <nehaniv@gmail.com>'}

eendebakpt_oapackage: returning=1, new=0
   new: set()

wojdyr_fityk: returning=1, new=3
   new: {'Marc Hahn <marc-benjamin.hahn@bam.de>', 'Marc Benjamin Hahn <mhahn1@NB1694.zit.bam.de>', 'MarcBHahn <60821053+MarcBHahn@users.noreply.github.com>'}

smearle_gym-city: returning=1, new=2
   new: {'Awesome-Aasim <65820963+Awesome-Aasim@users.noreply.github.com>', 'WeiShi <ws2297@nyu.edu>'}

acclab_dabestr: returning=0, new=3
   new: {'sunroofgod <e0725238@u.nus.edu>', 'Lian Kah Seng <90504567+sunroofgod@users.noreply.github.com>', 'Sangyu Xu <xusangyu@gmail.com>'}

ncoudray_deeppath: returning=1, new=3
   new: {'Coudray <coud

In [6]:
part3 = {
    'covid19datahub_covid19dev': dict(
        BeforeThemes='Automated bot contributions:Documentation updates',
        AfterThemes='Automated bot contributions:Bug fixes',
        RecentThemes='Automated bot contributions:Documentation updates',
        HypothesizedGapReason='Before the gap, all 10 commits were automated "Built site" builds with no human development; the first post-gap commits are "Fix workflow" and "Update pkgdown.yaml", suggesting the site-build CI broke and nothing was committed until it was repaired.',
        HasRecovered='Yes',
        WhyRecovered='Maintainer fixed the CI workflow and resumed data fixes ("Fix CHE", "Fix BEL"); automated site builds then continued (3,378 commits after the gap).',
        WhoRecovered='Same core maintainer(s)',
        Notes=''),
    'echonet_dynamic': dict(
        BeforeThemes='Dependency updates:Automated bot contributions',
        AfterThemes='Dependency updates:Automated bot contributions',
        RecentThemes='Automated bot contributions:Other',
        HypothesizedGapReason='Research code no longer actively developed after publication; pre-gap commits are almost entirely Dependabot version bumps plus "hacked together code for personal use", showing no ongoing human development.',
        HasRecovered='No',
        WhyRecovered='Post-gap activity is Dependabot bumps (tqdm, torch, pillow, scikit-learn) and a LICENSE file, which is automated upkeep, not renewed development (only 15 commits after the 24-month gap).',
        WhoRecovered='N/A - mostly Dependabot',
        Notes=''),
    'opengeospatial_ogc-geosparql': dict(
        BeforeThemes='Automated bot contributions:Documentation updates',
        AfterThemes='Documentation updates:Automated bot contributions',
        RecentThemes='Automated bot contributions:Documentation updates',
        HypothesizedGapReason='Short 2-month pause after finishing a review cycle of the standard; pre-gap commits fix the revision history table and add "history for full review", suggesting a draft was submitted for review.',
        HasRecovered='Yes',
        WhyRecovered='Editorial work resumed after review ("Editorial Updates", "Elaborate on length function", sample datasets added to homepage), with gh-pages deploys continuing.',
        WhoRecovered='Same contributors',
        Notes=''),
    'egri-nagy_sgpdec': dict(
        BeforeThemes='Documentation updates:Feature development',
        AfterThemes='Feature development:Bug fixes',
        RecentThemes='Documentation updates:Other',
        HypothesizedGapReason='Pre-gap commits are README, install info and bibliography updates (correcting the LNCS article title), suggesting the project was wrapped up after publication and the maintainer paused for 13 months.',
        HasRecovered='Yes',
        WhyRecovered='Recovery began with a license PR from the gap-packages organization (#9) followed by new functionality ("Dot Semigroup Action ... much need functionality"), tied to integrating sgpdec into the GAP package ecosystem.',
        WhoRecovered='Mix - original maintainer plus GAP packages contributors',
        Notes=''),
    'eendebakpt_oapackage': dict(
        BeforeThemes='Bug fixes:Release',
        AfterThemes='Bug fixes:Other',
        RecentThemes='Release:Bug fixes',
        HypothesizedGapReason='Natural post-release pause: the last pre-gap commits merge "New release 2.6.6" after build fixes; the 7-month gap (2020-01 to 2020-07) followed the release.',
        HasRecovered='Yes',
        WhyRecovered='Maintainer returned for code cleanup and compiler and style fixes (clang, swig, codacy), leading to later releases (2.7.14, 2.7.15).',
        WhoRecovered='Same maintainer (eendebakpt)',
        Notes=''),
    'wojdyr_fityk': dict(
        BeforeThemes='Automated bot contributions:Feature development',
        AfterThemes='Feature development:Dependency updates',
        RecentThemes='Automated bot contributions:Documentation updates',
        HypothesizedGapReason='Pre-gap activity was small (a log-scale button, spelling fixes, session saving) with several unmerged PR merge refs, consistent with a single maintainer having limited time.',
        HasRecovered='Yes',
        WhyRecovered='An external contributor PR improved Shirley background subtraction for XPS analysis, and the maintainer updated build compatibility (Lua 5.4 for Fedora Rawhide, wxGTK3, Sphinx).',
        WhoRecovered='New contributor (MarcBHahn) plus original maintainer',
        Notes=''),
    'smearle_gym-city': dict(
        BeforeThemes='Feature development:Dependency updates',
        AfterThemes='Other:Documentation updates',
        RecentThemes='Other:Documentation updates',
        HypothesizedGapReason='Research project ended: pre-gap commits are experiment tweaks (micro-rct metrics, HPC log_dir, Game of Life visualization), then 41 months of no commits, the longest gap of any project.',
        HasRecovered='Partially',
        WhyRecovered='Post-gap commits work on getting the old code running again via a Dockerfile ("Dockerfile works reliably, with rendering") and a merge into a "Wei" branch, suggesting a revival attempt; only 11 commits followed.',
        WhoRecovered='Original author, possibly with a new collaborator (Wei branch)',
        Notes=''),
    'acclab_dabestr': dict(
        BeforeThemes='Feature development:Documentation updates',
        AfterThemes='Feature development:Documentation updates',
        RecentThemes='Feature development:Release',
        HypothesizedGapReason='After adding the mini-meta function, docs/vignettes and a version bump, development stalled for 18 months, likely because effort shifted to the Python version of DABEST.',
        HasRecovered='Yes',
        WhyRecovered='A full "dabestr rebuild" porting "main functionalities from python ver" (PR #142), followed by new features and CRAN release work.',
        WhoRecovered='New contributor (sunroofgod) drove the rebuild',
        Notes=''),
    'aol_moloch': dict(
        BeforeThemes='N/A', AfterThemes='N/A',
        RecentThemes='Feature development:Automated bot contributions',
        HypothesizedGapReason='N/A active Project',
        HasRecovered='N/A', WhyRecovered='N/A', WhoRecovered='N/A',
        Notes=''),
    'ncoudray_deeppath': dict(
        BeforeThemes='Feature development:Automated bot contributions',
        AfterThemes='Dependency updates:Documentation updates',
        RecentThemes='Automated bot contributions:Dependency updates',
        HypothesizedGapReason='Short 3-month pause after a burst of research features (Vahadane normalization, melanoma, QuPath support), consistent with the author shifting focus after a research milestone.',
        HasRecovered='Partially',
        WhyRecovered='Author returned for performance work and README updates ("speed up HeatMap", "code optimization"), but most post-gap commits are Dependabot security bumps (tensorflow-gpu, django, pillow, urllib3).',
        WhoRecovered='Original author plus Dependabot',
        Notes=''),
}

In [7]:
for col in ['BeforeThemes', 'AfterThemes', 'HypothesizedGapReason', 'HasRecovered',
            'WhyRecovered', 'WhoRecovered', 'RecentThemes', 'Notes']:
    stats[col] = stats['Project'].map(lambda p: part3.get(p, {}).get(col, ''))

stats.to_csv(f"{netid}_project_stats.csv", sep=";", index=False)
stats[['Project', 'Currentstatus', 'BeforeThemes', 'AfterThemes', 'HasRecovered']]

,Project,Currentstatus,BeforeThemes,AfterThemes,HasRecovered
0,covid19datahub_covid19dev,Active,Automated bot contributions:Documentation updates,Automated bot contributions:Bug fixes,Yes
1,echonet_dynamic,Active,Dependency updates:Automated bot contributions,Dependency updates:Automated bot contributions,No
2,opengeospatial_ogc-geosparql,Active,Automated bot contributions:Documentation updates,Documentation updates:Automated bot contributions,Yes
3,egri-nagy_sgpdec,Active,Documentation updates:Feature development,Feature development:Bug fixes,Yes
4,eendebakpt_oapackage,Active,Bug fixes:Release,Bug fixes:Other,Yes
5,wojdyr_fityk,Active,Automated bot contributions:Feature development,Feature development:Dependency updates,Yes
6,smearle_gym-city,Active,Feature development:Dependency updates,Other:Documentation updates,Partially
7,acclab_dabestr,Active,Feature development:Documentation updates,Feature development:Documentation updates,Yes
8,aol_moloch,Active,N/A,N/A,N/A
9,ncoudray_deeppath,Inactive,Feature development:Automated bot contributions,Dependency updates:Documentation updates,Partially


In [8]:
who = {
    'covid19datahub_covid19dev': 'Same maintainer (1 returning author, 0 new)',
    'echonet_dynamic': 'Original lead David Ouyang and Ashwin Nayak merged Dependabot PRs; no new development (1 returning, 2 new identities)',
    'opengeospatial_ogc-geosparql': 'Same contributors (2 returning, 0 new)',
    'egri-nagy_sgpdec': 'Mix: 2 returning authors plus Max Horn (GAP packages maintainer) and co-author Chrystopher Nehaniv',
    'eendebakpt_oapackage': 'Same maintainer eendebakpt (1 returning, 0 new)',
    'wojdyr_fityk': 'Original maintainer plus new external contributor Marc Hahn (BAM), who appears under 3 identities',
    'smearle_gym-city': 'Original author plus new contributors WeiShi (NYU) and Awesome-Aasim',
    'acclab_dabestr': 'Entirely new contributors: sunroofgod/Lian Kah Seng (one person) and Sangyu Xu; 0 returning authors',
    'aol_moloch': 'N/A',
    'ncoudray_deeppath': 'Original author Coudray (same person, different cluster hosts) plus dependabot[bot]',
}
stats['WhoRecovered'] = stats['Project'].map(who)
stats.to_csv(f"{netid}_project_stats.csv", sep=";", index=False)

In [9]:
reflections = {
'covid19datahub_covid19dev': ("""
**Inactivity pattern:** Declining. Activity peaked during the early pandemic (2020-2021) and then settled into mostly automated site builds. There was only one gap of 3+ months, from 2022-11 to 2023-01 (3 months).

**Interpretability:** Moderately easy. All 10 commits before the gap are automated "Built site for COVID19" commits with no human development. The first commits after the gap are "Fix workflow" and "Update pkgdown.yaml", which points to a broken CI build.

**Likely reasons:** The project had already moved into maintenance mode. When the automated site-build workflow broke, commits stopped completely. The same maintainer repaired the workflow and fixed country data (CHE, BEL), and automated builds then resumed, producing 3,378 more commits.
""", "Declining project that moved into automated maintenance mode after the pandemic peak. Its single 3-month gap (2022-11 to 2023-01) likely came from a broken site-build workflow, since the first post-gap commits fix the workflow and pkgdown config. The same maintainer recovered it, and automated builds resumed."),

'echonet_dynamic': ("""
**Inactivity pattern:** Irregular, with 4 gaps of 3+ months and a sparse history (178 commits). The longest gap lasted 24 months (2023-01 to 2024-12).

**Interpretability:** Easy. Before the gap, nearly every commit is a Dependabot version bump, plus one human commit reading "hacked together code for personal use". After the gap, it's more Dependabot bumps (torch, pillow, scikit-learn), a LICENSE file, and PR merge refs.

**Likely reasons:** This is research code that accompanied a publication, and active development ended after its release. The post-gap activity isn't a real recovery. It's bot-driven dependency upkeep merged by the original lead (David Ouyang) and Ashwin Nayak, with only 15 commits after the gap.
""", "Irregular research codebase with 4 gaps of 3+ months; the longest lasted 24 months (2023-01 to 2024-12). Commits on both sides of the gap are mostly Dependabot bumps, so the project never truly recovered. Development effectively ended after publication."),

'opengeospatial_ogc-geosparql': ("""
**Inactivity pattern:** Cyclical, with bursts tied to standards review and release cycles. It never went 3+ months without commits, and its longest pause was 2 months (2023-01 to 2023-02).

**Interpretability:** Easy. Just before the pause, commits fix the revision history table and add "history for full review", showing a draft being finalized for review. After the pause come editorial updates, sample datasets, and homepage improvements.

**Likely reasons:** The pause was a natural wait while the standard was under review, and the same two contributors resumed editorial work afterward. This is a healthy, consistently active project with continuous gh-pages deployments.
""", "Cyclical standards project with 0 gaps of 3+ months; its longest pause was 2 months (2023-01 to 2023-02). Pre-pause commits prepare a draft for full review, and the same contributors resumed editorial work afterward. The project is consistently active and release-driven."),

'egri-nagy_sgpdec': ("""
**Inactivity pattern:** Irregular over a 13-year lifespan (2012-2025), with 11 gaps of 3+ months. The longest gap lasted 13 months (2018-01 to 2019-01).

**Interpretability:** Moderately easy. Before the gap, commits are mostly README, install info, and bibliography updates (correcting the LNCS article title), which looks like a project being wrapped up. After the gap, recovery starts with a license PR from the gap-packages organization (#9), followed by new functionality ("Dot Semigroup Action").

**Likely reasons:** The maintainer paused after the associated publication. The project came back as part of the GAP package ecosystem: Max Horn (a GAP packages maintainer) and co-author Chrystopher Nehaniv joined the 2 returning authors.
""", "Irregular, long-lived academic project with 11 gaps of 3+ months; the longest lasted 13 months (2018-01 to 2019-01). It paused after documentation and publication wrap-up. It recovered through integration into the GAP packages ecosystem, with new contributors Max Horn and Chrystopher Nehaniv."),

'eendebakpt_oapackage': ("""
**Inactivity pattern:** Cyclical, with development waves tied to releases over 10 years and 5 gaps of 3+ months. The longest gap lasted 7 months (2020-01 to 2020-07).

**Interpretability:** Easy. The last commits before the gap are build fixes and the merge of "New release 2.6.6". After the gap, commits are code cleanup and compiler/style fixes (clang, swig, codacy).

**Likely reasons:** This was a natural post-release pause, possibly extended by the onset of COVID-19 in early 2020. The same single maintainer (eendebakpt) returned with maintenance work that led to later releases (2.7.14, 2.7.15).
""", "Cyclical, release-driven project with 5 gaps of 3+ months; the longest lasted 7 months (2020-01 to 2020-07), right after release 2.6.6. The same maintainer returned with cleanup and compiler fixes. The pattern reflects normal post-release pauses."),

'wojdyr_fityk': ("""
**Inactivity pattern:** Cyclical over a 22-year lifespan (2003-2025), with the most gaps of any project: 14 gaps of 3+ months. The longest lasted 9 months (2019-08 to 2020-04).

**Interpretability:** Moderately hard. Before the gap, commits are small (a log-scale button, spelling fixes, session saving) and include unmerged PR refs, but nothing explicitly explains the stop.

**Likely reasons:** The project is mature and mostly maintained by one person with limited time. It recovered when an external contributor, Marc Hahn (BAM), submitted improvements to Shirley background subtraction for XPS analysis. The original maintainer also updated build compatibility (Lua 5.4 for Fedora Rawhide, wxGTK3, Sphinx).
""", "Mature, cyclical project with 14 gaps of 3+ months, the most of any project; the longest lasted 9 months (2019-08 to 2020-04). The cause was hard to pin down from commits, but it fits a single maintainer with limited time. It recovered through an external contributor's XPS feature and maintainer build-compatibility updates."),

'smearle_gym-city': ("""
**Inactivity pattern:** Declining, with 2 gaps of 3+ months, including the longest gap of any project: 41 months (2021-11 to 2025-03).

**Interpretability:** Easy. Before the gap, commits are research experiment tweaks (micro-rct metrics, HPC log_dir, Game of Life visualization). After the gap, commits focus on getting the old code running again with a Dockerfile.

**Likely reasons:** The research project ended and the author moved on. The recovery is only partial: 11 commits from the original author plus new contributors WeiShi (NYU) and Awesome-Aasim, mostly environment setup rather than new research.
""", "Declining research project with the longest gap of any project, 41 months (2021-11 to 2025-03), and 2 gaps of 3+ months overall. It went inactive when the research ended. The partial recovery was mostly Dockerfile work to revive the code, with new contributors WeiShi and Awesome-Aasim."),

'acclab_dabestr': ("""
**Inactivity pattern:** U-shaped. Activity was strong early, followed by an 18-month gap (2022-01 to 2023-06), then renewed activity. It had 4 gaps of 3+ months.

**Interpretability:** Easy. Before the gap, commits added the mini-meta function, vignettes, and a version bump. After the gap, commits are "dabestr rebuild" and "add main functionalities from python ver".

**Likely reasons:** Development likely shifted to the Python version of DABEST, which left the R package idle. It recovered through a complete rebuild by entirely new contributors (sunroofgod/Lian Kah Seng and Sangyu Xu), with 0 returning authors. That was followed by new features and CRAN release work (327 commits after the gap).
""", "U-shaped project with an 18-month gap (2022-01 to 2023-06) and 4 gaps of 3+ months. It stalled after a release while effort likely moved to the Python version. It was revived by a full rebuild from entirely new contributors (0 returning authors)."),

'aol_moloch': ("""
**Inactivity pattern:** Rising. This is the largest project (12,825 commits, 292 authors, 7,509 stars). It never had a month without commits, so it has 0 gaps of 3+ months and no longest gap.

**Interpretability:** Not applicable, since there was no inactivity to interpret.

**Activity patterns:** Activity grew as adoption and the contributor base increased. Recent commits show active feature development (a new summary page with D3.js charts and export functionality) and AI-assisted contributions co-authored with Copilot, which reflects modern development practices.
""", "Rising, continuously active project with 0 months of inactivity and therefore no gaps. It is by far the largest project (12,825 commits, 292 authors). Recent commits show ongoing feature development and Copilot-assisted contributions."),

'ncoudray_deeppath': ("""
**Inactivity pattern:** U-shaped, with 2 gaps of 3+ months. The longest lasted 3 months (2021-01 to 2021-03).

**Interpretability:** Moderately easy. Before the gap, there is a burst of research features (Vahadane normalization, melanoma, QuPath support). After the gap, there is performance work ("speed up HeatMap") and README updates, but most commits are Dependabot security bumps.

**Likely reasons:** The author shifted focus after a research milestone. The recovery was partial: the original author (Coudray, committing from different cluster hosts) returned briefly, and the rest of the activity is dependabot[bot]. The last GitHub commit was 2024-09-24, so the project is currently inactive.
""", "U-shaped research project with 2 gaps of 3+ months; the longest lasted 3 months (2021-01 to 2021-03), after a burst of research features. The recovery was partial, mostly Dependabot security bumps plus brief performance work from the original author. The project is currently inactive (last commit 2024-09-24)."),
}

for prj, (body, note) in reflections.items():
    with open(f"{netid}_reflection_{prj}.md", "w") as f:
        f.write(f"# Reflection: {prj}\n{body}")

stats['Notes'] = stats['Project'].map(lambda p: reflections[p][1])
stats.to_csv(f"{netid}_project_stats.csv", sep=";", index=False)
print("Wrote", len(reflections), "reflections")

Wrote 10 reflections


,Project,lastGHCommitDate,Currentstatus
0,covid19datahub_covid19dev,2025-06-30,Active
1,echonet_dynamic,2026-05-01,Active
2,opengeospatial_ogc-geosparql,2026-09-21,Active
3,egri-nagy_sgpdec,2026-06-30,Active
4,eendebakpt_oapackage,2026-02-04,Active
5,wojdyr_fityk,2026-03-02,Active
6,smearle_gym-city,2025-04-04,Active
7,acclab_dabestr,2026-07-29,Active
8,aol_moloch,2026-09-28,Active
9,ncoudray_deeppath,2024-09-24,Inactive
